# 02 Acquire: download, record provenance, version
Download your team's release from the SEC, record each file's size and SHA-256 hash, and track the data with DVC so a grader gets exactly your bytes.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "params.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from lookthrough import paths, acquire, load, mapping, holdings, clean, reconcile, report, tables, pipeline
P = paths.load_params(ROOT)
if not (ROOT / ".notebook-scratch").exists():
    raise RuntimeError("This teaching notebook writes outputs. Run tools/run_notebooks.py in its isolated scratch copy; use dvc repro for submission outputs.")
RDIR = paths.release_dir(P, ROOT)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
print("release folder:", RDIR.relative_to(ROOT), "| fixture mode:", P["data"]["use_fixture"])

In [ ]:
if P["data"]["use_fixture"]:
    files = sorted(RDIR.rglob("*.tsv"))
    mpath = ROOT / "data" / "manifest_fixture.json"
    rows = acquire.write_manifest(files, mpath, url="synthetic fixture", root=ROOT)
    print("Fixture mode: manifest written for", len(rows), "files (no download)")
else:
    raw = ROOT / P["data"]["raw_dir"]
    z = raw / f"{P['release']}_bdc.zip"
    if not z.exists():
        z = acquire.download_release(P["release"], raw, P["sec"]["user_agent"], P["sec"]["base_url"])
    out = acquire.unzip_release(z, raw, P["release"])          # always lands in data/raw/<release>_bdc
    mpath = ROOT / "data" / "manifest.json"
    rows = acquire.write_manifest([z], mpath, url=acquire.release_url(P["release"], P["sec"]["base_url"]), root=ROOT)
    print(f"Downloaded {z.name} ({rows[0]['bytes']:,} bytes) and unpacked it into {out.relative_to(ROOT)}")
pd.DataFrame(rows)[["path", "bytes", "sha256"]]

In [ ]:
print("Manifest verifies:", acquire.verify_manifest(mpath, root=ROOT) == [])

## Switch to your real release
1. Set `sec.user_agent` and `release` in `params.yaml`, and set `data.use_fixture: false`.
2. Re-run this notebook: it downloads `<release>_bdc.zip` from the SEC and unpacks it into `data/raw/<release>_bdc/`.
3. Version the data (in a terminal, inside your `cs2b/` folder):
```
dvc init                      # only if your repository does not use DVC yet
dvc add data/raw              # commit data/raw.dvc and data/manifest.json
git add data/raw.dvc data/manifest.json .gitignore && git commit -m "Add BDC release"
dvc push                      # to your team's DVC remote
```
Manifest paths are relative to the kit folder, so the manifest verifies on any clone. Never commit raw files to Git.